<a href="https://colab.research.google.com/github/lenhattung/UnsupervisedLearning/blob/main/Clustering_00_Feture_Scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

In [ ]:
# Tạo dữ liệu mẫu với scale khác nhau
np.random.seed(42)
data = pd.DataFrame({
    'tuoi': np.random.randint(20, 60, 100),  # Scale 20-60
    'thu_nhap': np.random.randint(10000, 100000, 100)  # Scale 10k-100k
})

In [ ]:
data.head(50)

In [ ]:
# Phân cụm KHÔNG có scaling
kmeans_no_scale = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_no_scale = kmeans_no_scale.fit_predict(data)

In [ ]:
# Phân cụm CÓ StandardScaler
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
kmeans_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_scaled = kmeans_scaled.fit_predict(data_scaled)

In [ ]:
print("Centroids không scale:")
print(kmeans_no_scale.cluster_centers_)
print("\nCentroids có scale:")
print(kmeans_scaled.cluster_centers_)

In [ ]:
# Thiết lập khung hình
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Hàm hỗ trợ vẽ Convex Hull ---
def draw_hull(ax, X, labels, title, xlabel, ylabel):
    # Vẽ các điểm dữ liệu
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50, alpha=0.6)

    # Lấy danh sách các nhãn cụm (0, 1, 2...)
    unique_labels = np.unique(labels)

    # Lấy bảng màu tương ứng để đường bao cùng màu với điểm
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

    for i, label in enumerate(unique_labels):
        # Lấy các điểm thuộc cụm hiện tại
        points = X[labels == label]

        # ConvexHull cần ít nhất 3 điểm để tạo hình đa giác
        if len(points) >= 3:
            hull = ConvexHull(points)

            # Vẽ các cạnh của bao lồi (đường viền đậm)
            for simplex in hull.simplices:
                ax.plot(points[simplex, 0], points[simplex, 1], color=colors[i], lw=2)

            # Tô màu nhạt bên trong vùng bao
            ax.fill(points[hull.vertices, 0], points[hull.vertices, 1], color=colors[i], alpha=0.1)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

# --- 1. Vẽ hình KHÔNG Scale ---
# Chuyển DataFrame thành numpy array để xử lý thống nhất
X_no_scale = data[['tuoi', 'thu_nhap']].values
draw_hull(ax1, X_no_scale, labels_no_scale,
          'K-Means: KHÔNG Scale\n(Vùng bao bị kéo dẹt theo trục ngang)',
          'Tuổi', 'Thu Nhập')

# --- 2. Vẽ hình CÓ Scale ---
# data_scaled đã là numpy array từ code trước của bạn
draw_hull(ax2, data_scaled, labels_scaled,
          'K-Means: CÓ StandardScaler\n(Các vùng bao đều và cân đối hơn)',
          'Tuổi (Scaled)', 'Thu Nhập (Scaled)')

plt.tight_layout()
plt.show()